# Car prediction using Machine Learning

# Introduction:

The price of a car depends on a lot of factors like the goodwill of the brand of the car,
features of the car, horsepower and the mileage it gives and many more. Car price
prediction is one of the major research areas in machine learning.

# Tools and Libraries:

- Python
- Jupyter Notebook
- Scikit-learn
- Pandas
- NumPy
- Matplotlib

In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, GridSearchCV, KFold

# Data Understanding:

In [2]:
df=pd.read_csv('car data.csv')
display(df)

,Car_Name,Year,Selling_Price,Present_Price,Driven_kms,Fuel_Type,Selling_type,Transmission,Owner
0,ritz,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,sx4,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0
2,ciaz,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0
3,wagon r,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0
4,swift,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0
...,...,...,...,...,...,...,...,...,...
296,city,2016,9.50,11.60,33988,Diesel,Dealer,Manual,0
297,brio,2015,4.00,5.90,60000,Petrol,Dealer,Manual,0
298,city,2009,3.35,11.00,87934,Petrol,Dealer,Manual,0
299,city,2017,11.50,12.50,9000,Diesel,Dealer,Manual,0


In [3]:
print('Checkingnthe missing values from the dataset: \n \n',df.isnull().sum())

Checkingnthe missing values from the dataset: 
 
 Car_Name         0
Year             0
Selling_Price    0
Present_Price    0
Driven_kms       0
Fuel_Type        0
Selling_type     0
Transmission     0
Owner            0
dtype: int64


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 301 entries, 0 to 300
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Car_Name       301 non-null    object 
 1   Year           301 non-null    int64  
 2   Selling_Price  301 non-null    float64
 3   Present_Price  301 non-null    float64
 4   Driven_kms     301 non-null    int64  
 5   Fuel_Type      301 non-null    object 
 6   Selling_type   301 non-null    object 
 7   Transmission   301 non-null    object 
 8   Owner          301 non-null    int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 21.3+ KB


In [5]:
objects_df=df.select_dtypes(include=['object'])
objects_col_names=objects_df.columns
print(objects_col_names)

Index(['Car_Name', 'Fuel_Type', 'Selling_type', 'Transmission'], dtype='object')


In [6]:
objects_df

,Car_Name,Fuel_Type,Selling_type,Transmission
0,ritz,Petrol,Dealer,Manual
1,sx4,Diesel,Dealer,Manual
2,ciaz,Petrol,Dealer,Manual
3,wagon r,Petrol,Dealer,Manual
4,swift,Diesel,Dealer,Manual
...,...,...,...,...
296,city,Diesel,Dealer,Manual
297,brio,Petrol,Dealer,Manual
298,city,Petrol,Dealer,Manual
299,city,Diesel,Dealer,Manual


In [7]:
numerical_df=df.select_dtypes(include=['int64','float64'])
numerical_col_names=numerical_df.columns
print(numerical_col_names)

Index(['Year', 'Selling_Price', 'Present_Price', 'Driven_kms', 'Owner'], dtype='object')


In [8]:
numerical_df

,Year,Selling_Price,Present_Price,Driven_kms,Owner
0,2014,3.35,5.59,27000,0
1,2013,4.75,9.54,43000,0
2,2017,7.25,9.85,6900,0
3,2011,2.85,4.15,5200,0
4,2014,4.60,6.87,42450,0
...,...,...,...,...,...
296,2016,9.50,11.60,33988,0
297,2015,4.00,5.90,60000,0
298,2009,3.35,11.00,87934,0
299,2017,11.50,12.50,9000,0


In [9]:
#Display summary of categorical features

for col in objects_col_names:
    print(f"\n{col} : \n{df[col].value_counts()}")
    


Car_Name : 
city                  26
corolla altis         16
verna                 14
fortuner              11
brio                  10
                      ..
KTM 390 Duke           1
Hero Passion X pro     1
TVS Jupyter            1
Honda CB Trigger       1
Hero  Ignitor Disc     1
Name: Car_Name, Length: 98, dtype: int64

Fuel_Type : 
Petrol    239
Diesel     60
CNG         2
Name: Fuel_Type, dtype: int64

Selling_type : 
Dealer        195
Individual    106
Name: Selling_type, dtype: int64

Transmission : 
Manual       261
Automatic     40
Name: Transmission, dtype: int64


In [10]:
#display the summary of numerical features
print(df[numerical_col_names].describe())

              Year  Selling_Price  Present_Price     Driven_kms       Owner
count   301.000000     301.000000     301.000000     301.000000  301.000000
mean   2013.627907       4.661296       7.628472   36947.205980    0.043189
std       2.891554       5.082812       8.642584   38886.883882    0.247915
min    2003.000000       0.100000       0.320000     500.000000    0.000000
25%    2012.000000       0.900000       1.200000   15000.000000    0.000000
50%    2014.000000       3.600000       6.400000   32000.000000    0.000000
75%    2016.000000       6.000000       9.900000   48767.000000    0.000000
max    2018.000000      35.000000      92.600000  500000.000000    3.000000


In [11]:
#Encoding 'previous_application.csv'
encoded_df= df.copy()

# Encoding different columns (With label encoder)
label_enc=['Fuel_Type','Selling_type','Transmission','Car_Name']
label_encoder = LabelEncoder()

for column in label_enc:
    encoded_value= label_encoder.fit_transform(encoded_df[column])
    encoded_df[column]= encoded_value

#Encoding different columns (with ordinal encoder)
all_mapping={'Year':{2003:0, 2004:1, 2005:2,2006:3,2007:4,2008:5,2009:6,2010:7,2011:8,2012:9,2013:10,2014:11,2015:12,2016:13,2017:14,2018:15}}

for column in all_mapping:
    mapping= all_mapping[column]
    ordinal_encoder= OrdinalEncoder(categories =[list(mapping.keys())])
    encoded_df[column]= ordinal_encoder.fit_transform(encoded_df[[column]])
    
    
display(encoded_df.head())

,Car_Name,Year,Selling_Price,Present_Price,Driven_kms,Fuel_Type,Selling_type,Transmission,Owner
0,90,11.0,3.35,5.59,27000,2,0,1,0
1,93,10.0,4.75,9.54,43000,1,0,1,0
2,68,14.0,7.25,9.85,6900,2,0,1,0
3,96,8.0,2.85,4.15,5200,2,0,1,0
4,92,11.0,4.60,6.87,42450,1,0,1,0


# Preprocessing

In [12]:
X=encoded_df.drop(columns=['Owner'])
y=encoded_df['Owner']

#Split the dataset
X_train, X_test, y_train, y_test= train_test_split(X,y,test_size=0.2, random_state=42)

#Train k-NN classifier

knn= KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

# Predict and evaluate

y_pred=knn.predict(X_test)
score=accuracy_score(y_test,y_pred)
print("Accuracy score withot scaling",score)

Accuracy score withot scaling 0.9836065573770492


- Apply Min-Max Scaling and Standardization to a dataset using scikit-learn
- Observe the effects of scaling on model performance by training a K-NN classifier before and after scaling

In [13]:
#Apply Min-max scaling
scaler= MinMaxScaler()
X_scaled= scaler.fit_transform(X)


In [14]:
#Split the scaled data
X_train_scaled, X_test_scaled, y_train_scaled, y_test_scaled= train_test_split(X_scaled,y,test_size=0.2,random_state=42)

#Train k-nn classifier on scaled data

knnScaled=KNeighborsClassifier(n_neighbors=5)
knnScaled.fit(X_train_scaled,y_train_scaled)

y_scaledPred=knnScaled.predict(X_test_scaled)

scaled_score=accuracy_score(y_test_scaled,y_scaledPred)

print("The accuracy score of a scaled data",scaled_score)

The accuracy score of a scaled data 0.9836065573770492


No improvement after scaling

In [15]:
#Apply standardization
scaler= StandardScaler()
X_std= scaler.fit_transform(X)

In [16]:
#Split the scaled data
X_train_std, X_test_std, y_train_std, y_test_std= train_test_split(X_std,y,test_size=0.2,random_state=42)

#Train k-nn classifier on scaled data

knnStd=KNeighborsClassifier(n_neighbors=5)
knnStd.fit(X_train_std,y_train_std)

y_stdPred=knnStd.predict(X_test_std)

std_score=accuracy_score(y_test_std,y_stdPred)

print("The accuracy score of a normalized data",std_score)

The accuracy score of a normalized data 0.9836065573770492


SyntaxError: invalid syntax (<ipython-input-17-0b024bbfe84e>, line 1)